In [ ]:
import albumentations as A
import cv2
import os
import numpy as np
from pathlib import Path

# Definimos la pipeline de augmentación "Best Practice" según la bibliografía
# Eliminamos rotaciones de 90 grados si las fotos son terrestres.
def get_train_transforms():
    return A.Compose([
        # Volteo horizontal: una naranja es simétrica, esto duplica los datos efectivos
        A.HorizontalFlip(p=0.5),

        # Rotación suave: simula el viento o ligeras inclinaciones de cámara (±15 grados)
        A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),

        # Variación de iluminación: Crucial para exteriores (sol/sombra)
        # RandomBrightnessContrast es más robusto que solo cambiar brillo
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),

        # Simulación de condiciones climáticas o desenfoque por movimiento del dron/vehículo
        A.OneOf([
            A.MotionBlur(p=1.0),
            A.GaussianBlur(p=1.0),
            A.GaussNoise(p=1.0),
        ], p=0.2),

        # Variación de color (Hue/Saturation): Para manejar diferentes estados de maduración
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),

    ], bbox_params=A.BboxParams(format='yolo', min_visibility=0.4, label_fields=['class_labels']))

# Función utilitaria para verificar tus augmentaciones antes de entrenar
# (Opcional: úsala para visualizar si las imágenes tienen sentido)
def visualize_augmentation(image_path, label_path):
    import matplotlib.pyplot as plt

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    with open(label_path, 'r') as f:
        labels = f.readlines()

    bboxes = []
    class_labels = []
    for label in labels:
        parts = list(map(float, label.strip().split()))
        class_labels.append(int(parts[0]))
        # Albumentations espera coordenadas normalizadas para formato 'yolo'
        bboxes.append(parts[1:])

    transform = get_train_transforms()
    try:
        transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
        transformed_image = transformed['image']

        plt.figure(figsize=(10,5))
        plt.subplot(1,2,1); plt.title("Original"); plt.imshow(image)
        plt.subplot(1,2,2); plt.title("Augmented"); plt.imshow(transformed_image)
        plt.show()
    except Exception as e:
        print(f"Error en augmentación visual: {e}")

if __name__ == "__main__":
    print("Módulo de augmentaciones listo. Importa 'get_train_transforms' en tu script de entrenamiento si usas un dataloader personalizado.")
    # Nota: YOLOv8 aplica sus propias augmentaciones internas muy eficientes.
    # Este script es útil si quieres pre-generar datos o modificar el dataloader.